# 01 · Chất lượng dữ liệu và đặc điểm
Mục tiêu: đo xem dữ liệu hiện có cho phép mô hình nào. Mọi số liệu được tính lại khi chạy notebook.

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from _common import *
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3})
news, px, tg = load()
vn30 = px[px.ticker == 'VN30'].set_index('date').sort_index()
print(f'news {len(news):,} | price rows {len(px):,} | targets {len(tg):,}')

## 1. Mật độ tin theo phiên (thị trường)

In [ ]:
per = news.dropna(subset=['session_date']).groupby('session_date').size()
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
per.rolling(20).mean().plot(ax=ax[0]); ax[0].axhline(20, color='r', ls='--', label='ngưỡng G1 = 20'); ax[0].set_title('Số bài/phiên (TB 20 phiên)'); ax[0].legend()
news.groupby([news.session_date.dt.year, 'source']).size().unstack(fill_value=0).plot.bar(stacked=True, ax=ax[1]); ax[1].set_title('Bài theo năm và nguồn')
savefig(fig, 'nb01_news_density'); plt.show()
print(per.groupby(per.index.year).agg(['median', 'min', 'max']))

## 2. Thời điểm đăng bài và quy tắc phiên (look-ahead)

In [ ]:
hr = news.published_at_ict.str[11:13].astype(int)
after = (news.published_at_ict.str[11:19] > '15:00:00').mean()
fig, ax = plt.subplots(figsize=(8, 3.5)); hr.value_counts().sort_index().plot.bar(ax=ax); ax.axvline(15, color='r'); ax.set_title('Giờ đăng (ICT)')
savefig(fig, 'nb01_hour'); plt.show()
print(f'Bài đăng sau 15:00 ICT: {after:.1%} (gán sang phiên kế tiếp); trước đó đăng sau 15:00 ICT (gán sang phiên kế tiếp); placeholder: {news.placeholder_ts.mean():.4%}')

## 3. Tin theo mã (độ thưa)

In [ ]:
import re
tick = sorted(px[px.in_vn30].ticker.unique())
pat = {t: re.compile(rf'\b{t}\b') for t in tick}
txt = news[['session_date', 'title']].dropna()
days = txt.session_date.nunique()
cov = {t: txt[txt.title.str.contains(pat[t])].session_date.nunique() / days for t in tick}
cov = pd.Series(cov).sort_values()
fig, ax = plt.subplots(figsize=(11, 4)); cov.plot.bar(ax=ax); ax.set_title('Tỷ lệ phiên có >=1 bài nhắc mã (tiêu đề)')
savefig(fig, 'nb01_ticker_cov'); plt.show()
print(f'trung vị {cov.median():.1%}; số mã <10% phiên: {(cov<.10).sum()}/{len(cov)}; >50%: {(cov>.5).sum()}')

## 4. Giá: phủ và chế độ biến động

In [ ]:
vol = np.log(vn30.close).diff().rolling(20).std() * np.sqrt(252)
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
vn30.close.plot(ax=ax[0]); ax[0].set_title('VN30')
vol.plot(ax=ax[1]); ax[1].set_title('Biến động thực hiện 20 phiên (annualised)')
savefig(fig, 'nb01_regimes'); plt.show()
yr = np.log(vn30.close).diff().groupby(vn30.index.year).agg(['mean', 'std', 'count'])
yr['mean_ann'] = yr['mean'] * 252; yr['vol_ann'] = yr['std'] * np.sqrt(252)
print(yr[['mean_ann', 'vol_ann', 'count']].round(3))

In [ ]:
cal = px[px.ticker=='VNINDEX'].date.nunique()
c = px[px.in_vn30].groupby('ticker').date.nunique()
member_days = px.groupby('date').in_vn30.sum()
print('ngày VNINDEX:', cal, '| ngày có đủ 30 thành viên có giá:', int((member_days==30).sum()), '| ngày 29 thành viên:', int((member_days==29).sum()))

## Kết luận từ notebook 01 (đọc cùng số liệu ở trên)
- Tin mức thị trường dày và đều; tin theo mã thưa, nên chỉ số sợ hãi thị trường là tín hiệu chính.
- Biến động thay đổi mạnh giữa các năm (chế độ thị trường), nên mô hình phải được kiểm tra qua walk-forward chứ không dựa vào một lần chia.
- Thành viên VN30 có giá đủ 30 mã trừ khoảng ROS.